# Import Libraries

In [1]:
import gc
gc.collect()

51

In [2]:
import numpy as np
import pandas as pd
import seaborn as sns
from matplotlib import pyplot as plt
import math
from sklearn.preprocessing import LabelEncoder
import os

from pathlib import Path
import warnings
# Suppress Python warnings (Scikit-Learn, Pandas, etc.)
warnings.filterwarnings("ignore")

# Load Dataset

In [3]:
# Project root is one level up from the Notebook folder
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "Notebook" else Path.cwd()
DATA_DIR   = PROJECT_ROOT / "Data"
MODEL_DIR  = PROJECT_ROOT / "Model"        

df = pd.read_csv(DATA_DIR / "Melbourne_housing.csv")

In [4]:
df.head()

,Suburb,Address,Rooms,Type,Method,SellerG,Date,Distance,Postcode,Bedroom,...,Landsize,BuildingArea,YearBuilt,CouncilArea,Latitude,Longitude,Regionname,Propertycount,ParkingArea,Price
0,Abbotsford,68 Studley St,2,h,SS,Jellis,3/9/2016,2.5,3067.0,2.0,...,126.0,inf,NaN,Yarra City Council,-37.8014,144.9958,Northern Metropolitan,4019.0,Carport,NaN
1,Airport West,154 Halsey Rd,3,t,PI,Nelson,3/9/2016,13.5,3042.0,3.0,...,303.0,225,2016.0,Moonee Valley City Council,-37.7180,144.8780,Western Metropolitan,3464.0,Detached Garage,840000.0
2,Albert Park,105 Kerferd Rd,2,h,S,hockingstuart,3/9/2016,3.3,3206.0,2.0,...,120.0,82,1900.0,Port Phillip City Council,-37.8459,144.9555,Southern Metropolitan,3280.0,Attached Garage,1275000.0
3,Albert Park,85 Richardson St,2,h,S,Thomson,3/9/2016,3.3,3206.0,2.0,...,159.0,inf,NaN,Port Phillip City Council,-37.8450,144.9538,Southern Metropolitan,3280.0,Indoor,1455000.0
4,Alphington,30 Austin St,3,h,SN,McGrath,3/9/2016,6.4,3078.0,3.0,...,174.0,122,2003.0,Darebin City Council,-37.7818,145.0198,Northern Metropolitan,2211.0,Parkade,NaN


In [5]:
df.tail()

,Suburb,Address,Rooms,Type,Method,SellerG,Date,Distance,Postcode,Bedroom,...,Landsize,BuildingArea,YearBuilt,CouncilArea,Latitude,Longitude,Regionname,Propertycount,ParkingArea,Price
34852,Reservoir,18 Elinda Pl,3,u,SP,RW,30/09/2017,12.0,3073.0,3.0,...,NaN,105.0,1990.0,Darebin City Council,-37.69769,145.02332,Northern Metropolitan,21650.0,Parkade,475000.0
34853,Roxburgh Park,14 Stainsby Cr,4,h,S,Raine,30/09/2017,20.6,3064.0,4.0,...,NaN,225.0,1995.0,Hume City Council,-37.63665,144.92976,Northern Metropolitan,5833.0,Underground,591000.0
34854,Springvale South,8 Bellbird Ct,4,h,PI,Barry,30/09/2017,22.2,3172.0,4.0,...,534.0,152.0,1970.0,Greater Dandenong City Council,-37.97037,145.15449,South-Eastern Metropolitan,4054.0,Carport,NaN
34855,Springvale South,30 Waddington Cr,3,h,S,Harcourts,30/09/2017,22.2,3172.0,3.0,...,544.0,NaN,NaN,Greater Dandenong City Council,-37.97751,145.14813,South-Eastern Metropolitan,4054.0,Detached Garage,780500.0
34856,Westmeadows,42 Pascoe St,4,h,S,Barry,30/09/2017,16.5,3049.0,4.0,...,813.0,140.0,1960.0,Hume City Council,-37.67631,144.89409,Northern Metropolitan,2474.0,Attached Garage,791000.0


# Create a Copy

In [6]:
df_model = df.copy()

# Checking Null values

In [7]:
df.isnull().sum()

Suburb               0
Address              0
Rooms                0
Type                 0
Method               0
SellerG              0
Date                 0
Distance             1
Postcode             1
Bedroom           8217
Bathroom          8226
Car               8728
Landsize         11810
BuildingArea     21097
YearBuilt        19306
CouncilArea          3
Latitude          7976
Longitude         7976
Regionname           0
Propertycount        3
ParkingArea          0
Price             7610
dtype: int64

In [8]:
df.shape

(34857, 22)

# Remove Target Missing Values

In [9]:
df = df.dropna(subset = 'Price')
df

,Suburb,Address,Rooms,Type,Method,SellerG,Date,Distance,Postcode,Bedroom,...,Landsize,BuildingArea,YearBuilt,CouncilArea,Latitude,Longitude,Regionname,Propertycount,ParkingArea,Price
1,Airport West,154 Halsey Rd,3,t,PI,Nelson,3/9/2016,13.5,3042.0,3.0,...,303.0,225,2016.0,Moonee Valley City Council,-37.71800,144.87800,Western Metropolitan,3464.0,Detached Garage,840000.0
2,Albert Park,105 Kerferd Rd,2,h,S,hockingstuart,3/9/2016,3.3,3206.0,2.0,...,120.0,82,1900.0,Port Phillip City Council,-37.84590,144.95550,Southern Metropolitan,3280.0,Attached Garage,1275000.0
3,Albert Park,85 Richardson St,2,h,S,Thomson,3/9/2016,3.3,3206.0,2.0,...,159.0,inf,NaN,Port Phillip City Council,-37.84500,144.95380,Southern Metropolitan,3280.0,Indoor,1455000.0
5,Alphington,6 Smith St,4,h,S,Brace,3/9/2016,6.4,3078.0,3.0,...,853.0,263,1930.0,Darebin City Council,-37.77070,145.03180,Northern Metropolitan,2211.0,Underground,2000000.0
6,Alphington,5/6 Yarralea St,3,h,S,Jellis,3/9/2016,6.4,3078.0,3.0,...,208.0,inf,2013.0,Darebin City Council,-37.78540,145.03250,Northern Metropolitan,2211.0,Outdoor Stall,1110000.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
34851,Noble Park,5 Blaby St,3,h,PI,C21,30/09/2017,22.7,3174.0,3.0,...,569.0,130.0,1959.0,Greater Dandenong City Council,-37.96900,145.18226,South-Eastern Metropolitan,11806.0,Indoor,627500.0
34852,Reservoir,18 Elinda Pl,3,u,SP,RW,30/09/2017,12.0,3073.0,3.0,...,NaN,105.0,1990.0,Darebin City Council,-37.69769,145.02332,Northern Metropolitan,21650.0,Parkade,475000.0
34853,Roxburgh Park,14 Stainsby Cr,4,h,S,Raine,30/09/2017,20.6,3064.0,4.0,...,NaN,225.0,1995.0,Hume City Council,-37.63665,144.92976,Northern Metropolitan,5833.0,Underground,591000.0
34855,Springvale South,30 Waddington Cr,3,h,S,Harcourts,30/09/2017,22.2,3172.0,3.0,...,544.0,NaN,NaN,Greater Dandenong City Council,-37.97751,145.14813,South-Eastern Metropolitan,4054.0,Detached Garage,780500.0


In [10]:
df.shape

(27247, 22)

# Drop Irrelevant Columns

In [11]:
df.columns

Index(['Suburb', 'Address', 'Rooms', 'Type', 'Method', 'SellerG', 'Date',
       'Distance', 'Postcode', 'Bedroom', 'Bathroom', 'Car', 'Landsize',
       'BuildingArea', 'YearBuilt', 'CouncilArea', 'Latitude', 'Longitude',
       'Regionname', 'Propertycount', 'ParkingArea', 'Price'],
      dtype='object')

In [12]:
df[['Suburb', 'Address', 'SellerG', 'Date', 'CouncilArea']]

,Suburb,Address,SellerG,Date,CouncilArea
1,Airport West,154 Halsey Rd,Nelson,3/9/2016,Moonee Valley City Council
2,Albert Park,105 Kerferd Rd,hockingstuart,3/9/2016,Port Phillip City Council
3,Albert Park,85 Richardson St,Thomson,3/9/2016,Port Phillip City Council
5,Alphington,6 Smith St,Brace,3/9/2016,Darebin City Council
6,Alphington,5/6 Yarralea St,Jellis,3/9/2016,Darebin City Council
...,...,...,...,...,...
34851,Noble Park,5 Blaby St,C21,30/09/2017,Greater Dandenong City Council
34852,Reservoir,18 Elinda Pl,RW,30/09/2017,Darebin City Council
34853,Roxburgh Park,14 Stainsby Cr,Raine,30/09/2017,Hume City Council
34855,Springvale South,30 Waddington Cr,Harcourts,30/09/2017,Greater Dandenong City Council


In [13]:
df = df.drop(columns = ['Suburb', 'Address', 'SellerG', 'Date', 'CouncilArea'])
df.head()

,Rooms,Type,Method,Distance,Postcode,Bedroom,Bathroom,Car,Landsize,BuildingArea,YearBuilt,Latitude,Longitude,Regionname,Propertycount,ParkingArea,Price
1,3,t,PI,13.5,3042.0,3.0,2.0,1.0,303.0,225,2016.0,-37.7180,144.8780,Western Metropolitan,3464.0,Detached Garage,840000.0
2,2,h,S,3.3,3206.0,2.0,1.0,0.0,120.0,82,1900.0,-37.8459,144.9555,Southern Metropolitan,3280.0,Attached Garage,1275000.0
3,2,h,S,3.3,3206.0,2.0,1.0,0.0,159.0,inf,NaN,-37.8450,144.9538,Southern Metropolitan,3280.0,Indoor,1455000.0
5,4,h,S,6.4,3078.0,3.0,2.0,4.0,853.0,263,1930.0,-37.7707,145.0318,Northern Metropolitan,2211.0,Underground,2000000.0
6,3,h,S,6.4,3078.0,3.0,2.0,2.0,208.0,inf,2013.0,-37.7854,145.0325,Northern Metropolitan,2211.0,Outdoor Stall,1110000.0


# Handle Missing Values

## Numerical Columns

In [14]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 27247 entries, 1 to 34856
Data columns (total 17 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Rooms          27247 non-null  int64  
 1   Type           27247 non-null  object 
 2   Method         27247 non-null  object 
 3   Distance       27246 non-null  float64
 4   Postcode       27246 non-null  float64
 5   Bedroom        20806 non-null  float64
 6   Bathroom       20800 non-null  float64
 7   Car            20423 non-null  float64
 8   Landsize       17982 non-null  float64
 9   BuildingArea   10670 non-null  object 
 10  YearBuilt      12084 non-null  float64
 11  Latitude       20993 non-null  float64
 12  Longitude      20993 non-null  float64
 13  Regionname     27247 non-null  object 
 14  Propertycount  27244 non-null  float64
 15  ParkingArea    27247 non-null  object 
 16  Price          27247 non-null  float64
dtypes: float64(11), int64(1), object(5)
memory usage: 3.7+ 

In [15]:
num_cols = []
cat_cols = []

for i in df.columns :
    if df[i].dtype != 'object' :
        num_cols.append(i)
    else :
        cat_cols.append(i)

num_cols

['Rooms',
 'Distance',
 'Postcode',
 'Bedroom',
 'Bathroom',
 'Car',
 'Landsize',
 'YearBuilt',
 'Latitude',
 'Longitude',
 'Propertycount',
 'Price']

In [16]:
for i in num_cols :
    df[i] = df[i].fillna(df[i].median())

# Handle Categorical Missing Values

In [17]:
cat_cols

['Type', 'Method', 'BuildingArea', 'Regionname', 'ParkingArea']

In [18]:
for i in cat_cols :
    df[i] = df[i].fillna(df[i].mode()[0])

In [19]:
df.isnull().sum()

Rooms            0
Type             0
Method           0
Distance         0
Postcode         0
Bedroom          0
Bathroom         0
Car              0
Landsize         0
BuildingArea     0
YearBuilt        0
Latitude         0
Longitude        0
Regionname       0
Propertycount    0
ParkingArea      0
Price            0
dtype: int64

# Fix BuildingArea

In [20]:
cat_cols.remove('BuildingArea')
cat_cols

['Type', 'Method', 'Regionname', 'ParkingArea']

In [21]:
df['BuildingArea']

1          225
2           82
3          inf
5          263
6          inf
         ...  
34851    130.0
34852    105.0
34853    225.0
34855      120
34856    140.0
Name: BuildingArea, Length: 27247, dtype: object

In [22]:
# Replace inf and -inf with NaN
df['BuildingArea'].replace(['inf', np.inf, -np.inf], np.nan, inplace=True)

In [23]:
df['BuildingArea'] = pd.to_numeric(df['BuildingArea'],errors='coerce')
df['BuildingArea'].head(50)

1     225.0
2      82.0
3       NaN
5     263.0
6       NaN
7     242.0
9     251.0
10    117.0
11      NaN
13     76.0
15      NaN
16    399.0
17      NaN
19      NaN
20      NaN
21    118.0
22    103.0
23    180.0
25    123.0
26    218.0
27    129.0
28    167.0
29    154.0
30    275.0
31      NaN
32      NaN
33      NaN
34      NaN
35      NaN
36    121.0
37    120.0
38    120.0
39    120.0
42    120.0
43    120.0
44    120.0
45    120.0
46    125.0
47    255.0
49    120.0
50     75.0
51    120.0
53    156.0
55    240.0
57    268.0
58    120.0
60    117.0
61    120.0
63    120.0
64    108.0
Name: BuildingArea, dtype: float64

In [24]:
df['BuildingArea'].fillna(df['BuildingArea'].median(), inplace =  True)
df['BuildingArea'].head(50)

1     225.0
2      82.0
3     120.0
5     263.0
6     120.0
7     242.0
9     251.0
10    117.0
11    120.0
13     76.0
15    120.0
16    399.0
17    120.0
19    120.0
20    120.0
21    118.0
22    103.0
23    180.0
25    123.0
26    218.0
27    129.0
28    167.0
29    154.0
30    275.0
31    120.0
32    120.0
33    120.0
34    120.0
35    120.0
36    121.0
37    120.0
38    120.0
39    120.0
42    120.0
43    120.0
44    120.0
45    120.0
46    125.0
47    255.0
49    120.0
50     75.0
51    120.0
53    156.0
55    240.0
57    268.0
58    120.0
60    117.0
61    120.0
63    120.0
64    108.0
Name: BuildingArea, dtype: float64

# Remove Duplicate Rows

In [25]:
df = df.drop_duplicates()
df

,Rooms,Type,Method,Distance,Postcode,Bedroom,Bathroom,Car,Landsize,BuildingArea,YearBuilt,Latitude,Longitude,Regionname,Propertycount,ParkingArea,Price
1,3,t,PI,13.5,3042.0,3.0,2.0,1.0,303.0,225.0,2016.0,-37.71800,144.87800,Western Metropolitan,3464.0,Detached Garage,840000.0
2,2,h,S,3.3,3206.0,2.0,1.0,0.0,120.0,82.0,1900.0,-37.84590,144.95550,Southern Metropolitan,3280.0,Attached Garage,1275000.0
3,2,h,S,3.3,3206.0,2.0,1.0,0.0,159.0,120.0,1970.0,-37.84500,144.95380,Southern Metropolitan,3280.0,Indoor,1455000.0
5,4,h,S,6.4,3078.0,3.0,2.0,4.0,853.0,263.0,1930.0,-37.77070,145.03180,Northern Metropolitan,2211.0,Underground,2000000.0
6,3,h,S,6.4,3078.0,3.0,2.0,2.0,208.0,120.0,2013.0,-37.78540,145.03250,Northern Metropolitan,2211.0,Outdoor Stall,1110000.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
34851,3,h,PI,22.7,3174.0,3.0,1.0,6.0,569.0,130.0,1959.0,-37.96900,145.18226,South-Eastern Metropolitan,11806.0,Indoor,627500.0
34852,3,u,SP,12.0,3073.0,3.0,1.0,1.0,512.0,105.0,1990.0,-37.69769,145.02332,Northern Metropolitan,21650.0,Parkade,475000.0
34853,4,h,S,20.6,3064.0,4.0,2.0,2.0,512.0,225.0,1995.0,-37.63665,144.92976,Northern Metropolitan,5833.0,Underground,591000.0
34855,3,h,S,22.2,3172.0,3.0,2.0,1.0,544.0,120.0,1970.0,-37.97751,145.14813,South-Eastern Metropolitan,4054.0,Detached Garage,780500.0


# Handle Outliers

In [26]:
def cap_outliers(df,col):
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5*IQR
    upper = Q3 + 1.5*IQR
    df[col] = df[col].clip(lower,upper)

for col in ['Price', 'BuildingArea', 'Landsize']:
    cap_outliers(df,col)

# Log Transform Skewed Features

In [27]:
col = ['Price', 'BuildingArea', 'Landsize']

for x in col :
    df[x] = np.log1p(df[x])

df.head()

,Rooms,Type,Method,Distance,Postcode,Bedroom,Bathroom,Car,Landsize,BuildingArea,YearBuilt,Latitude,Longitude,Regionname,Propertycount,ParkingArea,Price
1,3,t,PI,13.5,3042.0,3.0,2.0,1.0,5.717028,4.795791,2016.0,-37.7180,144.8780,Western Metropolitan,3464.0,Detached Garage,13.641158
2,2,h,S,3.3,3206.0,2.0,1.0,0.0,4.795791,4.795791,1900.0,-37.8459,144.9555,Southern Metropolitan,3280.0,Attached Garage,14.058458
3,2,h,S,3.3,3206.0,2.0,1.0,0.0,5.075174,4.795791,1970.0,-37.8450,144.9538,Southern Metropolitan,3280.0,Indoor,14.190517
5,4,h,S,6.4,3078.0,3.0,2.0,4.0,6.749931,4.795791,1930.0,-37.7707,145.0318,Northern Metropolitan,2211.0,Underground,14.508658
6,3,h,S,6.4,3078.0,3.0,2.0,2.0,5.342334,4.795791,2013.0,-37.7854,145.0325,Northern Metropolitan,2211.0,Outdoor Stall,13.919871


In [28]:
df.columns

Index(['Rooms', 'Type', 'Method', 'Distance', 'Postcode', 'Bedroom',
       'Bathroom', 'Car', 'Landsize', 'BuildingArea', 'YearBuilt', 'Latitude',
       'Longitude', 'Regionname', 'Propertycount', 'ParkingArea', 'Price'],
      dtype='object')

# One-Hot Encode Categories

In [29]:
ohe_cols = [
    'Type',
    'Method',
    'Regionname',
    'ParkingArea'
]

In [30]:
for i in ohe_cols :
    print(df[i].value_counts(), '\n')

Type
h    18468
u     5907
t     2864
Name: count, dtype: int64 

Method
S     17511
SP     3599
PI     3255
VB     2684
SA      190
Name: count, dtype: int64 

Regionname
Southern Metropolitan         8523
Northern Metropolitan         7862
Western Metropolitan          5814
Eastern Metropolitan          3272
South-Eastern Metropolitan    1340
Eastern Victoria               166
Northern Victoria              166
Western Victoria                96
Name: count, dtype: int64 

ParkingArea
Carport            4947
Attached Garage    4858
Detached Garage    4857
Indoor             4439
Parkade            3659
Underground        2033
Outdoor Stall      1619
Parking Pad         827
Name: count, dtype: int64 



In [31]:
df = pd.get_dummies(df, columns=ohe_cols, drop_first=True)
df.head()

,Rooms,Distance,Postcode,Bedroom,Bathroom,Car,Landsize,BuildingArea,YearBuilt,Latitude,...,Regionname_Southern Metropolitan,Regionname_Western Metropolitan,Regionname_Western Victoria,ParkingArea_Carport,ParkingArea_Detached Garage,ParkingArea_Indoor,ParkingArea_Outdoor Stall,ParkingArea_Parkade,ParkingArea_Parking Pad,ParkingArea_Underground
1,3,13.5,3042.0,3.0,2.0,1.0,5.717028,4.795791,2016.0,-37.7180,...,False,True,False,False,True,False,False,False,False,False
2,2,3.3,3206.0,2.0,1.0,0.0,4.795791,4.795791,1900.0,-37.8459,...,True,False,False,False,False,False,False,False,False,False
3,2,3.3,3206.0,2.0,1.0,0.0,5.075174,4.795791,1970.0,-37.8450,...,True,False,False,False,False,True,False,False,False,False
5,4,6.4,3078.0,3.0,2.0,4.0,6.749931,4.795791,1930.0,-37.7707,...,False,False,False,False,False,False,False,False,False,True
6,3,6.4,3078.0,3.0,2.0,2.0,5.342334,4.795791,2013.0,-37.7854,...,False,False,False,False,False,False,True,False,False,False


In [32]:
tmp = df['Price']
df.drop(columns = ['Price'], inplace = True)
df.columns

Index(['Rooms', 'Distance', 'Postcode', 'Bedroom', 'Bathroom', 'Car',
       'Landsize', 'BuildingArea', 'YearBuilt', 'Latitude', 'Longitude',
       'Propertycount', 'Type_t', 'Type_u', 'Method_S', 'Method_SA',
       'Method_SP', 'Method_VB', 'Regionname_Eastern Victoria',
       'Regionname_Northern Metropolitan', 'Regionname_Northern Victoria',
       'Regionname_South-Eastern Metropolitan',
       'Regionname_Southern Metropolitan', 'Regionname_Western Metropolitan',
       'Regionname_Western Victoria', 'ParkingArea_Carport',
       'ParkingArea_Detached Garage', 'ParkingArea_Indoor',
       'ParkingArea_Outdoor Stall', 'ParkingArea_Parkade',
       'ParkingArea_Parking Pad', 'ParkingArea_Underground'],
      dtype='object')

In [33]:
tmp

1        13.641158
2        14.058458
3        14.190517
5        14.508658
6        13.919871
           ...    
34851    13.349501
34852    13.071072
34853    13.289573
34855    13.567691
34856    13.581055
Name: Price, Length: 27239, dtype: float64

# Label Encoding

In [34]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 27239 entries, 1 to 34856
Data columns (total 32 columns):
 #   Column                                 Non-Null Count  Dtype  
---  ------                                 --------------  -----  
 0   Rooms                                  27239 non-null  int64  
 1   Distance                               27239 non-null  float64
 2   Postcode                               27239 non-null  float64
 3   Bedroom                                27239 non-null  float64
 4   Bathroom                               27239 non-null  float64
 5   Car                                    27239 non-null  float64
 6   Landsize                               27239 non-null  float64
 7   BuildingArea                           27239 non-null  float64
 8   YearBuilt                              27239 non-null  float64
 9   Latitude                               27239 non-null  float64
 10  Longitude                              27239 non-null  float64
 11  Propert

In [35]:
df.head()

,Rooms,Distance,Postcode,Bedroom,Bathroom,Car,Landsize,BuildingArea,YearBuilt,Latitude,...,Regionname_Southern Metropolitan,Regionname_Western Metropolitan,Regionname_Western Victoria,ParkingArea_Carport,ParkingArea_Detached Garage,ParkingArea_Indoor,ParkingArea_Outdoor Stall,ParkingArea_Parkade,ParkingArea_Parking Pad,ParkingArea_Underground
1,3,13.5,3042.0,3.0,2.0,1.0,5.717028,4.795791,2016.0,-37.7180,...,False,True,False,False,True,False,False,False,False,False
2,2,3.3,3206.0,2.0,1.0,0.0,4.795791,4.795791,1900.0,-37.8459,...,True,False,False,False,False,False,False,False,False,False
3,2,3.3,3206.0,2.0,1.0,0.0,5.075174,4.795791,1970.0,-37.8450,...,True,False,False,False,False,True,False,False,False,False
5,4,6.4,3078.0,3.0,2.0,4.0,6.749931,4.795791,1930.0,-37.7707,...,False,False,False,False,False,False,False,False,False,True
6,3,6.4,3078.0,3.0,2.0,2.0,5.342334,4.795791,2013.0,-37.7854,...,False,False,False,False,False,False,True,False,False,False


In [36]:
bool_cols = df.columns[df.dtypes == 'bool'].tolist()
bool_cols

['Type_t',
 'Type_u',
 'Method_S',
 'Method_SA',
 'Method_SP',
 'Method_VB',
 'Regionname_Eastern Victoria',
 'Regionname_Northern Metropolitan',
 'Regionname_Northern Victoria',
 'Regionname_South-Eastern Metropolitan',
 'Regionname_Southern Metropolitan',
 'Regionname_Western Metropolitan',
 'Regionname_Western Victoria',
 'ParkingArea_Carport',
 'ParkingArea_Detached Garage',
 'ParkingArea_Indoor',
 'ParkingArea_Outdoor Stall',
 'ParkingArea_Parkade',
 'ParkingArea_Parking Pad',
 'ParkingArea_Underground']

In [37]:
le = LabelEncoder()
le

LabelEncoder()

In [38]:
for i in bool_cols :
    df[i] = le.fit_transform(df[i])

In [39]:
df.head()

,Rooms,Distance,Postcode,Bedroom,Bathroom,Car,Landsize,BuildingArea,YearBuilt,Latitude,...,Regionname_Southern Metropolitan,Regionname_Western Metropolitan,Regionname_Western Victoria,ParkingArea_Carport,ParkingArea_Detached Garage,ParkingArea_Indoor,ParkingArea_Outdoor Stall,ParkingArea_Parkade,ParkingArea_Parking Pad,ParkingArea_Underground
1,3,13.5,3042.0,3.0,2.0,1.0,5.717028,4.795791,2016.0,-37.7180,...,0,1,0,0,1,0,0,0,0,0
2,2,3.3,3206.0,2.0,1.0,0.0,4.795791,4.795791,1900.0,-37.8459,...,1,0,0,0,0,0,0,0,0,0
3,2,3.3,3206.0,2.0,1.0,0.0,5.075174,4.795791,1970.0,-37.8450,...,1,0,0,0,0,1,0,0,0,0
5,4,6.4,3078.0,3.0,2.0,4.0,6.749931,4.795791,1930.0,-37.7707,...,0,0,0,0,0,0,0,0,0,1
6,3,6.4,3078.0,3.0,2.0,2.0,5.342334,4.795791,2013.0,-37.7854,...,0,0,0,0,0,0,1,0,0,0


# Add Price columns again

In [40]:
df['Price'] = tmp

In [41]:
df

,Rooms,Distance,Postcode,Bedroom,Bathroom,Car,Landsize,BuildingArea,YearBuilt,Latitude,...,Regionname_Western Metropolitan,Regionname_Western Victoria,ParkingArea_Carport,ParkingArea_Detached Garage,ParkingArea_Indoor,ParkingArea_Outdoor Stall,ParkingArea_Parkade,ParkingArea_Parking Pad,ParkingArea_Underground,Price
1,3,13.5,3042.0,3.0,2.0,1.0,5.717028,4.795791,2016.0,-37.71800,...,1,0,0,1,0,0,0,0,0,13.641158
2,2,3.3,3206.0,2.0,1.0,0.0,4.795791,4.795791,1900.0,-37.84590,...,0,0,0,0,0,0,0,0,0,14.058458
3,2,3.3,3206.0,2.0,1.0,0.0,5.075174,4.795791,1970.0,-37.84500,...,0,0,0,0,1,0,0,0,0,14.190517
5,4,6.4,3078.0,3.0,2.0,4.0,6.749931,4.795791,1930.0,-37.77070,...,0,0,0,0,0,0,0,0,1,14.508658
6,3,6.4,3078.0,3.0,2.0,2.0,5.342334,4.795791,2013.0,-37.78540,...,0,0,0,0,0,1,0,0,0,13.919871
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
34851,3,22.7,3174.0,3.0,1.0,6.0,6.345636,4.795791,1959.0,-37.96900,...,0,0,0,0,1,0,0,0,0,13.349501
34852,3,12.0,3073.0,3.0,1.0,1.0,6.240276,4.795791,1990.0,-37.69769,...,0,0,0,0,0,0,1,0,0,13.071072
34853,4,20.6,3064.0,4.0,2.0,2.0,6.240276,4.795791,1995.0,-37.63665,...,0,0,0,0,0,0,0,0,1,13.289573
34855,3,22.2,3172.0,3.0,2.0,1.0,6.300786,4.795791,1970.0,-37.97751,...,0,0,0,1,0,0,0,0,0,13.567691


In [42]:
df = df.astype(float)

In [43]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 27239 entries, 1 to 34856
Data columns (total 33 columns):
 #   Column                                 Non-Null Count  Dtype  
---  ------                                 --------------  -----  
 0   Rooms                                  27239 non-null  float64
 1   Distance                               27239 non-null  float64
 2   Postcode                               27239 non-null  float64
 3   Bedroom                                27239 non-null  float64
 4   Bathroom                               27239 non-null  float64
 5   Car                                    27239 non-null  float64
 6   Landsize                               27239 non-null  float64
 7   BuildingArea                           27239 non-null  float64
 8   YearBuilt                              27239 non-null  float64
 9   Latitude                               27239 non-null  float64
 10  Longitude                              27239 non-null  float64
 11  Propert

In [44]:
for i in df.columns :
    print(df[i].nunique(), end = " ")

12 213 209 14 10 13 928 1 151 11366 12275 336 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2478 

In [45]:
df.shape

(27239, 33)

In [46]:
df.columns

Index(['Rooms', 'Distance', 'Postcode', 'Bedroom', 'Bathroom', 'Car',
       'Landsize', 'BuildingArea', 'YearBuilt', 'Latitude', 'Longitude',
       'Propertycount', 'Type_t', 'Type_u', 'Method_S', 'Method_SA',
       'Method_SP', 'Method_VB', 'Regionname_Eastern Victoria',
       'Regionname_Northern Metropolitan', 'Regionname_Northern Victoria',
       'Regionname_South-Eastern Metropolitan',
       'Regionname_Southern Metropolitan', 'Regionname_Western Metropolitan',
       'Regionname_Western Victoria', 'ParkingArea_Carport',
       'ParkingArea_Detached Garage', 'ParkingArea_Indoor',
       'ParkingArea_Outdoor Stall', 'ParkingArea_Parkade',
       'ParkingArea_Parking Pad', 'ParkingArea_Underground', 'Price'],
      dtype='object')

# Preprocessed dataset

In [47]:
DATA_DIR.mkdir(exist_ok=True)
df.to_csv(DATA_DIR / "Cleaned_dataset.csv", index=False)

In [48]:
# output_folder = 'D:\Linear-Regression-project-with-Streamlit\Data'
# output_filename = 'Cleaned_dataset.csv'
# file_path = os.path.join(output_folder, output_filename)
# os.makedirs(output_folder, exist_ok=True)
# df.to_csv(file_path, index=False)